In [ ]:
import torch
import torchaudio
import numpy as np
import tensorflow as tf

# =========================
# CONFIG
# =========================
SAMPLE_RATE = 16000
N_MFCC = 40
N_FFT = 400
HOP_LENGTH = 160
FIXED_LENGTH = 16000

In [ ]:
labels = [
    "down", "go", "left", "no", "off", "on",
    "right", "silence", "stop", "unknown",
    "up", "yes"
]

In [ ]:
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    melkwargs={
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "n_mels": 40
    }
)

def pad_trim(waveform):
    if waveform.shape[1] < FIXED_LENGTH:
        pad = FIXED_LENGTH - waveform.shape[1]
        waveform = torch.nn.functional.pad(waveform, (0, pad))
    else:
        waveform = waveform[:, :FIXED_LENGTH]
    return waveform

In [ ]:
data = torch.load("PATH_TO_MFCC_PT_FILE")

X_train = data["X_train"].unsqueeze(1).float()

mean = X_train.mean()
std  = X_train.std()

In [ ]:
def extract_features(file_path):
    wav, sr = torchaudio.load(file_path)

    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

    wav = pad_trim(wav)

    mfcc = mfcc_transform(wav).squeeze(0)  # (40, frames)

    # normalize (IMPORTANT)
    mfcc = (mfcc - mean) / std

    return mfcc

In [ ]:
interpreter = tf.lite.Interpreter(model_path="PATH_TO_TFLITE_INT8_MODEL")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("INPUT:", input_details)
print("OUTPUT:", output_details)

INPUT: [{'name': 'serving_default_input_1:0', 'index': 0, 'shape': array([  1,  40, 101,   1], dtype=int32), 'shape_signature': array([ -1,  40, 101,   1], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.09317018836736679, 65), 'quantization_parameters': {'scales': array([0.09317019], dtype=float32), 'zero_points': array([65], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
OUTPUT: [{'name': 'StatefulPartitionedCall:0', 'index': 17, 'shape': array([ 1, 12], dtype=int32), 'shape_signature': array([-1, 12], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.08099570125341415, 15), 'quantization_parameters': {'scales': array([0.0809957], dtype=float32), 'zero_points': array([15], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [ ]:
def quantize_input(x_float, scale, zero_point):
    x_int8 = x_float / scale + zero_point
    x_int8 = np.round(x_int8).astype(np.int8)
    return x_int8

def dequantize_output(x_int8, scale, zero_point):
    return scale * (x_int8 - zero_point)

In [ ]:
def prepare_input(mfcc):
    x = mfcc.unsqueeze(0).unsqueeze(0).numpy().astype(np.float32)


    input_shape = input_details[0]['shape']


    if input_shape[1] == 40:  # NHWC: [1,40,T,1]
        x = np.transpose(x, (0,2,3,1))

    return x

In [ ]:
def predict(file_path):
    mfcc = extract_features(file_path)
    x_float = prepare_input(mfcc)

    # get quant params
    scale, zero_point = input_details[0]['quantization']

    # quantize input
    x_int8 = quantize_input(x_float, scale, zero_point)

    interpreter.set_tensor(input_details[0]['index'], x_int8)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])

    # dequantize output
    out_scale, out_zero = output_details[0]['quantization']
    output = dequantize_output(output, out_scale, out_zero)

    pred_idx = np.argmax(output)
    confidence = float(np.max(output))

    return labels[pred_idx], confidence, output

In [ ]:
file_path = "PATH_TO_SAMPLE_WAV_FILE"

pred, conf, raw = predict(file_path)

print("Prediction :", pred)
print("Confidence:", conf)

Prediction : down
Confidence: 3.563810855150223


In [ ]:
def top_k(output, k=5):
    idx = np.argsort(output[0])[::-1][:k]
    return [(labels[i], float(output[0][i])) for i in idx]

print(top_k(raw))

[('down', 3.563810855150223), ('unknown', -1.052944116294384), ('no', -1.3769269213080406), ('stop', -1.538918323814869), ('yes', -1.9438968300819397)]


In [ ]:
import numpy as np

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

probs = softmax(raw[0])

# Top-K with probabilities
def top_k_probs(probs, k=5):
    idx = np.argsort(probs)[::-1][:k]
    return [(labels[i], float(probs[i])) for i in idx]

print(top_k_probs(probs))

[('down', 0.9699020287534368), ('unknown', 0.009587307551788578), ('no', 0.006934141742048646), ('stop', 0.005897130555882298), ('yes', 0.003933333872497659)]


In [ ]:
mfcc = extract_features(file_path)
x_float = prepare_input(mfcc)

scale, zero_point = input_details[0]['quantization']
x_int8 = quantize_input(x_float, scale, zero_point)

In [ ]:
np.save("INPUT.npy", x_int8)

In [ ]:
def to_c_array(x):
    return ", ".join(map(str, x.flatten()))

print(to_c_array(x_int8))

-34, -33, -34, -32, -31, -32, -33, -33, -30, -20, -11, -13, -17, -19, -19, -22, -26, -28, -28, -29, -29, -29, -29, -30, -29, -30, -32, -31, -31, -31, -32, -33, -34, -32, -32, -28, -23, -20, 1, 13, 25, 35, 45, 50, 52, 53, 54, 54, 54, 53, 53, 51, 51, 50, 49, 47, 44, 42, 40, 38, 34, 33, 33, 33, 30, 28, 26, 22, 20, 19, 16, 15, 13, 11, 6, 5, 4, 4, 2, 0, 0, 0, -2, -1, -5, -9, -8, -9, -11, -11, -13, -16, -16, -16, -15, -16, -18, -20, -21, -18, -14, 68, 69, 68, 70, 70, 70, 69, 69, 66, 53, 51, 54, 59, 62, 64, 65, 64, 65, 67, 64, 68, 68, 69, 69, 70, 70, 68, 69, 70, 69, 69, 69, 69, 70, 70, 75, 81, 84, 89, 95, 97, 98, 96, 94, 92, 90, 89, 89, 89, 90, 90, 91, 92, 91, 91, 92, 92, 93, 94, 95, 96, 98, 98, 98, 100, 100, 100, 101, 101, 100, 101, 100, 99, 97, 96, 94, 92, 91, 90, 90, 90, 91, 88, 87, 85, 85, 86, 84, 82, 82, 80, 79, 79, 80, 80, 80, 78, 76, 74, 72, 69, 66, 67, 67, 67, 65, 66, 66, 67, 67, 73, 70, 69, 68, 69, 69, 68, 69, 69, 66, 65, 67, 65, 65, 65, 66, 65, 64, 64, 66, 65, 65, 67, 67, 67, 67, 71

In [ ]:
wav_files = [
    "/SAMPLE_AUDIOFILES_1.wav",
    "/SAMPLE_AUDIOFILES_2.wav",
    "/SAMPLE_AUDIOFILES_3.wav",
    "/SAMPLE_AUDIOFILES_4.wav",
    "/SAMPLE_AUDIOFILES_5.wav",
    "/SAMPLE_AUDIOFILES_6.wav",
    "/SAMPLE_AUDIOFILES_7.wav",
    "/SAMPLE_AUDIOFILES_8.wav",
    "/SAMPLE_AUDIOFILES_9.wav",
    "/SAMPLE_AUDIOFILES_10.wav",
    "/SAMPLE_AUDIOFILES_11.wav",
]

In [ ]:
import os

def predict_batch(wav_files):
    results = []
    correct = 0

    print("="*60)

    for file_path in wav_files:
        pred, conf, raw = predict(file_path)

        # Extract ground truth from folder name
        true_label = os.path.basename(os.path.dirname(file_path))

        is_correct = (pred == true_label)
        if is_correct:
            correct += 1

        results.append((file_path, true_label, pred, conf, is_correct))

        print(f"\nFile       : {os.path.basename(file_path)}")
        print(f"True Label : {true_label}")
        print(f"Predicted  : {pred}")
        print(f"Confidence : {conf:.4f}")
        print(f"Status     : {'✅ CORRECT' if is_correct else '❌ WRONG'}")

    accuracy = correct / len(wav_files)

    print("\n" + "="*60)
    print(f"Batch Accuracy: {accuracy:.4f} ({correct}/{len(wav_files)})")

    return results

In [ ]:
results = predict_batch(wav_files)


File       : 0132a06d_nohash_1.wav
True Label : yes
Predicted  : yes
Confidence : 8.5045
Status     : ✅ CORRECT

File       : 0585b66d_nohash_2.wav
True Label : yes
Predicted  : yes
Confidence : 7.6136
Status     : ✅ CORRECT

File       : 0132a06d_nohash_1.wav
True Label : up
Predicted  : up
Confidence : 6.9656
Status     : ✅ CORRECT

File       : 0135f3f2_nohash_1.wav
True Label : up
Predicted  : up
Confidence : 2.1059
Status     : ✅ CORRECT

File       : 0165e0e8_nohash_2.wav
True Label : up
Predicted  : off
Confidence : 0.4860
Status     : ❌ WRONG

File       : 412c675c_nohash_0.wav
True Label : left
Predicted  : left
Confidence : 4.2928
Status     : ✅ CORRECT

File       : 40738a2d_nohash_0.wav
True Label : left
Predicted  : go
Confidence : 0.0810
Status     : ❌ WRONG

File       : 3ec05c3d_nohash_0.wav
True Label : left
Predicted  : left
Confidence : 3.0778
Status     : ✅ CORRECT

File       : 0132a06d_nohash_2.wav
True Label : down
Predicted  : down
Confidence : 3.5638
Status   

In [ ]:
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(model_path="PATH_TO_TFLITE_INT8")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(input_details)
print(output_details)

[{'name': 'serving_default_input_1:0', 'index': 0, 'shape': array([  1,  40, 101,   1], dtype=int32), 'shape_signature': array([ -1,  40, 101,   1], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.09317018836736679, 65), 'quantization_parameters': {'scales': array([0.09317019], dtype=float32), 'zero_points': array([65], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
[{'name': 'StatefulPartitionedCall:0', 'index': 17, 'shape': array([ 1, 12], dtype=int32), 'shape_signature': array([-1, 12], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.08099570125341415, 15), 'quantization_parameters': {'scales': array([0.0809957], dtype=float32), 'zero_points': array([15], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
input_index  = input_details[0]["index"]
output_index = output_details[0]["index"]
input_dtype  = input_details[0]["dtype"]

In [ ]:
print(input_dtype)

<class 'numpy.int8'>


In [ ]:
print(input_details[0]["quantization"])

(0.09317018836736679, 65)


In [ ]:
scale, zero_point = input_details[0]["quantization"]

In [ ]:
print(scale)
print(zero_point)

0.09317018836736679
65


In [ ]:
input_dtype == np.int8

True

In [ ]:
input_dtype == np.uint8

False

In [ ]:
input_dtype == np.int8

True

In [ ]:
import numpy as np
import tensorflow as tf

def predict_tflite_single(tflite_path, mfcc, mean, std):
    # -----------------------
    # Load interpreter
    # -----------------------
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_index  = input_details[0]["index"]
    output_index = output_details[0]["index"]
    input_dtype  = input_details[0]["dtype"]

    # -----------------------
    # PREPARE INPUT
    # -----------------------
    x = mfcc.numpy()  # torch → numpy

    # normalize (IMPORTANT: same as PyTorch)
    x = (x - mean) / std

    # add batch + channel: (1,1,40,98)
    x = x[None, None, :, :]

    # NHWC conversion (IMPORTANT for your CNN TFLite)
    x = np.transpose(x, (0, 2, 3, 1))

    # -----------------------
    # QUANTIZATION HANDLING
    # -----------------------
    if input_dtype == np.int8 or input_dtype == np.uint8:
        scale, zero_point = input_details[0]["quantization"]
        x = x / scale + zero_point
        x = np.round(x)

        if input_dtype == np.int8:
            x = x.astype(np.int8)
        else:
            x = x.astype(np.uint8)
    else:
        x = x.astype(np.float32)

    # -----------------------
    # RUN INFERENCE
    # -----------------------
    interpreter.set_tensor(input_index, x)
    interpreter.invoke()

    output = interpreter.get_tensor(output_index)

    # -----------------------
    # DEQUANTIZE OUTPUT
    # -----------------------
    if output_details[0]["dtype"] in [np.int8, np.uint8]:
        scale, zero_point = output_details[0]["quantization"]
        output = (output.astype(np.float32) - zero_point) * scale

    probs = tf.nn.softmax(output).numpy()
    pred = np.argmax(probs)

    return pred, probs

In [ ]:
pred, probs = predict_tflite_single(
    "/content/model_int8.tflite",
    mfcc,
    mean,
    std
)

print("Pred:", pred)
print("Probs:", probs)

Pred: 11
Probs: [[4.4991857e-06 1.4476719e-06 3.1430940e-05 3.5286266e-06 8.6008495e-06
  2.5521233e-06 6.0084876e-05 6.4403588e-07 4.0076106e-05 1.1486110e-04
  3.1068902e-07 9.9973196e-01]]


deployment level

In [ ]:
wav_files=[
"/content/drive/MyDrive/KWS/yes/01bb6a2a_nohash_3.wav",
"/content/drive/MyDrive/KWS/yes/01648c51_nohash_1.wav",
"/content/drive/MyDrive/KWS/yes/03431e13_nohash_0.wav",
"/content/drive/MyDrive/KWS/yes/099d52ad_nohash_0.wav",
]

In [ ]:
import torch
import torchaudio
import numpy as np
import tensorflow as tf

In [ ]:
# -----------------------
# CONFIG (same as training)
# -----------------------
SAMPLE_RATE = 16000
N_MFCC = 40
N_FFT = 400
HOP_LENGTH = 160
FIXED_LENGTH = 16000

# -----------------------
# MFCC Transform
# -----------------------
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    melkwargs={
        'n_fft': N_FFT,
        'hop_length': HOP_LENGTH,
        'n_mels': 40
    }
)

# -----------------------
# Pad / Trim (same!)
# -----------------------
def pad_trim(waveform):
    if waveform.shape[1] < FIXED_LENGTH:
        pad = FIXED_LENGTH - waveform.shape[1]
        waveform = torch.nn.functional.pad(waveform, (0, pad))
    else:
        waveform = waveform[:, :FIXED_LENGTH]
    return waveform

# -----------------------
# SINGLE FILE FUNCTION
# -----------------------
def wav_to_mfcc(path):
    waveform, sr = torchaudio.load(path)

    # Resample if needed
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

    # Pad/trim
    waveform = pad_trim(waveform)

    # MFCC
    mfcc = mfcc_transform(waveform)

    # Remove channel dim (same as training)
    mfcc = mfcc.squeeze(0)

    return mfcc

# -----------------------
# USE IT
# -----------------------
mfcc3 = wav_to_mfcc("/content/drive/MyDrive/KWS/yes/03431e13_nohash_0.wav")

print(mfcc3.shape)   # should match training shape (40, ~98)

torch.Size([40, 101])


In [ ]:
mean=-4.1967
std=34.9636

In [ ]:
x3 = mfcc3.numpy()

In [ ]:
x3 = (x3 - mean) / std

In [ ]:
x3 = x3[None, None, :, :]

In [ ]:
x3 = np.transpose(x3, (0, 2, 3, 1))

In [ ]:
print(x3.shape)

(1, 40, 101, 1)


In [ ]:
interpreter = tf.lite.Interpreter(model_path="PATH_TO_TFLITE_INT8")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(input_details)
print(output_details)

[{'name': 'serving_default_input_1:0', 'index': 0, 'shape': array([  1,  40, 101,   1], dtype=int32), 'shape_signature': array([ -1,  40, 101,   1], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.09317018836736679, 65), 'quantization_parameters': {'scales': array([0.09317019], dtype=float32), 'zero_points': array([65], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
[{'name': 'StatefulPartitionedCall:0', 'index': 17, 'shape': array([ 1, 12], dtype=int32), 'shape_signature': array([-1, 12], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.08099570125341415, 15), 'quantization_parameters': {'scales': array([0.0809957], dtype=float32), 'zero_points': array([15], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [ ]:
input_details[0]["quantization"]

(0.09317018836736679, 65)

In [ ]:
x3 = x3 / 0.09317018836736679 + 65
x3 = np.round(x3)
x3 = x3.astype(np.int8)

In [ ]:
import numpy as np
import tensorflow as tf

def predict_tflite_clean(tflite_path, x):
    # -----------------------
    # Load interpreter
    # -----------------------
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_index  = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]

    # -----------------------
    # RUN INFERENCE
    # -----------------------
    interpreter.set_tensor(input_index, x)
    interpreter.invoke()

    output = interpreter.get_tensor(output_index)[0]

    # -----------------------
    # DEQUANTIZE OUTPUT
    # -----------------------
    pred = int(np.argmax(output))

    return pred, output

In [ ]:
pred3, probs3 = predict_tflite_clean(
    "/content/model_int8.tflite",
    x3
)

print("Pred:", pred3)
print("Probs:", probs3)

Pred: 11
Probs: [ -9 -12  28  23 -31 -21  -3 -45 -52  13 -14  43]


In [ ]:
def save_as_c_array_int8(x_int8, filename="mfcc_input_yes.h", var_name="model_input"):
    x_flat = x_int8.flatten()

    with open(filename, "w") as f:
        f.write("#include <stdint.h>\n\n")
        f.write("#ifndef MODEL_INPUT_NAME\n\n")
        f.write("#define MODEL_INPUT_NAME model_input\n\n")
        f.write("#endif\n\n")
        f.write(f"const int8_t {var_name}[{len(x_flat)}] = {{\n")

        for i, val in enumerate(x_flat):
            f.write(f"{int(val)}, ")
            if (i + 1) % 16 == 0:
                f.write("\n")

        f.write("\n};\n")

    print(f"Saved: {filename}")

In [ ]:
save_as_c_array_int8(x3,"input_keyword.h","MODEL_INPUT_NAME")

Saved: input_keyword_5.h
